## All Packages

In [ ]:
import warnings
from torch import nn
from collections import deque
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from numpy.ma.extras import average
import torch
import random
from IPython.display import clear_output
from ipywidgets import IntProgress
from IPython.display import display as display_bar
import time
import math

## Game Deck Class

In [ ]:
class Deck:
    """
    This handles the game env and RL learning functions. 
    All functions are explained except for hidden function with 
    respect to the class and the get functions which I believed 
    to be self-explanatory
    """
    def __init__(self, decks, penetration, device):
        self.device = device 
        self.decks = decks #Given Model is trained for 6
        self.penetration = penetration #Given model is trained for 0.87
        self.cards = []
        self.running_count = 0  # Was calculating every time with a function a variable helps w runtime
        self.card_counts = {}    # Dictionary also saves runtime had a function that computed from a list iteration with was not efficient
        
        self._fresh_shoe()

    ###########################
    # State functions, functions to help build state tensor
    ###########################

    def get_score_and_soft(self, hand):
        total, aces = 0, 0
        for card in hand:
            if card == 'A':
                aces += 1
                total += 11
            else:
                total += card
        while total > 21 and aces > 0:
            total -= 10
            aces -= 1
        return total, int(aces > 0)

    def get_running_count(self):
        return self.running_count

    def get_true_count(self):
        decks_remaining = len(self.cards) / 52
        divisor = max(decks_remaining, 1/52)
        return self.running_count / divisor

    def get_hidden_prob(self):
        total = len(self.cards)
        if total == 0:
            return [0] * 10
        
        probs = [self.card_counts[i]/total for i in range(2, 11)]
        probs.append(self.card_counts['A']/total)
        return probs

    def get_state_action(self):
        """
        Creates the tensor for the action models input layer
        """
        #Asses action based on the deck of current index
        idx = min(self.hand_idx, len(self.hands) - 1); player_total, is_soft = self.get_score_and_soft(self.hands[idx])
        
        player_total_norm = player_total / 30 #Normalize over max total
        dealer_up = self._card_value(self.dealer[0]) / 11 if self.dealer else 0 #Normalize over max dealer up
        true_count_norm = self.get_true_count()/8.0 #Got to 0.8 through trail and error, soft form of normalization
        hidden_prob = self.get_hidden_prob() # Normalized list of prob of each value being the hidden card based on cards seen so far in deck

        return torch.tensor(hidden_prob + [self.stake / 10,player_total_norm,float(is_soft),dealer_up,true_count_norm], dtype=torch.float, device=self.device).unsqueeze(0)

    def get_state_bet(self):
        """
        Creates the tensor for the bet models input layer
        """
        
        true_count_norm = self.get_true_count()/8.0 #Got to 0.8 through trail and error, soft form of normalization
        hidden_prob = self.get_hidden_prob()

        return torch.tensor(hidden_prob + [true_count_norm], dtype=torch.float, device=self.device).unsqueeze(0)

    ###########################
    # Game logic, functions to help run the game
    ###########################

    def _reveal(self, card):
        if card in [2, 3, 4, 5, 6]:
            self.running_count += 1
        elif card in [10, 'A']:
            self.running_count -= 1

    def deal(self):
        """
        Deals a card in O(1) time, improved over previous versions
        """
        c = self.cards.pop()
        self.card_counts[c] -= 1
        return c

    def run_dealer(self):
        """
        Deals until real game stop conditions
        """
        self._reveal(self.secret)
        self.dealer.append(self.secret)  
        while True:
            score, _ = self.get_score_and_soft(self.dealer)
            if score >= 17:
                break
            c = self.deal()
            self.dealer.append(c)
            self._reveal(c)

    def execute(self, action):
        """
        Handles the possible actions and rules of the game 
        0: Hit 
        1: Double 
        2: Stand 
        3: Split
        """
        hand = self.hands[self.hand_idx]
        
        if action == 0:  # HIT
            c = self.deal(); hand.append(c); self._reveal(c)
            if self.get_score_and_soft(hand)[0] > 21: self._next_hand() #End hand if bust/go to next hand
        elif action == 1:  # DOUBLE
            if len(hand) == 2:
                #Deal next card, double stake, and go to next hand
                c = self.deal(); hand.append(c); self._reveal(c); self.stakes[self.hand_idx] *= 2; self._next_hand()
            else: #If the model tries to double when it should not add a penalty 
                self.illegal = -10; self._next_hand(); self.done = True
        elif action == 2:  # STAND
            self._next_hand()
        elif action == 3:  # SPLIT
            if len(hand) == 2 and hand[0] == hand[1]: #If two cards and the same can split 
                c1, c2 = hand
                # Dealing two new cards
                nc1 = self.deal(); nc2 = self.deal()
                self._reveal(nc1); self._reveal(nc2)
                # Create the two separate hands
                self.hands[self.hand_idx] = [c1, nc1]
                self.hands.insert(self.hand_idx + 1, [c2, nc2])
                self.stakes.insert(self.hand_idx + 1, self.stakes[self.hand_idx])
                #If an ace split end the hands for the two aces after the deal
                if c1 == "A" and  c2 == "A": self._next_hand(); self._next_hand();                     
            else: #if try to split illegally add penalty
                self._next_hand(); self.illegal = -10; self.done = True     

    def _next_hand(self):
        self.hand_idx += 1
        if self.hand_idx >= len(self.hands):
            self.done = True

    def reward(self):
        """
        Gets rewards based off of each player hand compared to 
        respective stake and dealer outcome
        """
        if self.illegal != 0:
            return self.illegal
        
        total_reward = 0
        d_score, _ = self.get_score_and_soft(self.dealer)
        for i, hand in enumerate(self.hands):
            p_score, _ = self.get_score_and_soft(hand)
            stake = self.stakes[i]
            if p_score > 21:
                total_reward -= stake
            elif d_score > 21 or p_score > d_score:
                total_reward += stake
            elif p_score < d_score:
                total_reward -= stake
        return total_reward

    ###########################
    # RL lifecycle/ Methods to run Deep RL
    ###########################

    def run_game(self, deep):
        """
        Run the RL training for one hand of black jack
        """

        #Check Deck Reset
        self._check_reset()

        #Reset Relevent Variables
        self.done = False; self.illegal = 0; self.hands = [[]]; self.stakes = [1]; self.hand_idx = 0; self.dealer = []; self.stake = 1

        #Get the random epsilon bet
        state = self.get_state_bet()
        if np.random.rand() < deep.epsilon_bet:
            bet_action = random.choice([0, 1, 2, 3])
        else:
            with torch.no_grad():
                bet_action = deep.bcModel(state).argmax().item()

        #Get the best size
        bet_sizes = [0.1, 0.5, 1.0, 10]; self.stake = bet_sizes[bet_action]; self.bet_state = state; self.bet_action = bet_action

        #Start the initial deal
        self._start_game(self.stake)

        while not self.done: #Until the game is over

            #Random epsilon action
            state = self.get_state_action()
            if np.random.rand() < deep.epsilon_action:
                action = random.choice([0, 1, 2, 3])
            else:
                with torch.no_grad():
                    action = deep.acModel(state).argmax().item()

            # Below am running to get relevant info for buffer
            prev_state = state; self.execute(action)

            if self.done and self.illegal == 0: self.run_dealer()
            
            next_state = self.get_state_action(); r = self.reward() / 15 if self.done else 0 #Normalized reward 
            
            deep.act_buffer.appendleft((prev_state, action, next_state, r, self.done)) #Add relevant into to action buffer
        
        deep.bet_buffer.appendleft((self.bet_state, self.bet_action, self.reward() / 15)) #Update reduced info (bc state terminated) to bet buffer

    def run_test(self, bet_model, action_model, count_history, total_bet):
        """
        Used as validation of performance of model
        """
        #Check Deck Reset
        self._check_reset()
        
        #Reset Relevant Variables
        self.done = False; self.illegal = 0; self.hands = [[]]; self.stakes = [1]; self.hand_idx = 0; self.dealer = []; self.stake = 1

        #Get best bet
        state = self.get_state_bet()
        with torch.no_grad(): bet_action = bet_model(state).argmax().item()
        bet_sizes = [0.1, 0.5, 1.0, 10]; self.stake = bet_sizes[bet_action]

        # For output display of testing
        true_count = round(self.get_true_count(), 2)   
        count_history.append((true_count, self.stake)) 

        #Start the initial deal
        self._start_game(self.stake)

        while not self.done:
            #Get action
            state = self.get_state_action()
            
            with torch.no_grad():
                action = action_model(state).argmax().item()

            #Update state
            self.execute(action)
            if self.done and self.illegal == 0: self.run_dealer()

        for each in self.stakes:
            total_bet.append(each)
            
            

    def _start_game(self, stake):
        self.hands = [[]]; self.stakes = [stake]
        self.hand_idx = 0; self.dealer = []; self.done = False

        c1 = self.deal(); self.hands[0].append(c1); self._reveal(c1)
        self.secret = self.deal() 
        c2 = self.deal(); self.hands[0].append(c2); self._reveal(c2)
        s = self.deal(); self.dealer.append(s); self._reveal(s)

        self._check_bj()

    def _check_bj(self):
        p_score, _ = self.get_score_and_soft(self.hands[0])
        if p_score == 21:                       
            self._reveal(self.secret)           
            d_score, _ = self.get_score_and_soft(self.dealer + [self.secret])
            if d_score == 21:
                self.stakes[0] = 0              
            else:
                self.stakes[0] *= 1.5           
            self.done = True

    def _check_reset(self):
        cut = int(52 * self.decks * (1 - self.penetration))
        if len(self.cards) < cut:
            self._fresh_shoe()

    def _card_value(self, card):
        return 11 if card == 'A' else card

    def _fresh_shoe(self):
        shoe = [10]*16*self.decks + [9]*4*self.decks + [8]*4*self.decks + \
               [7]*4*self.decks + [6]*4*self.decks + [5]*4*self.decks + \
               [4]*4*self.decks + [3]*4*self.decks + [2]*4*self.decks + ['A']*4*self.decks
        random.shuffle(shoe)
        self.cards = shoe
        
        self.running_count = 0
        self.count = []
        self.card_counts = {i: 4 * self.decks for i in range(2, 10)}
        self.card_counts[10] = 16 * self.decks
        self.card_counts['A'] = 4 * self.decks
        return shoe

## Import NNs and DQL Learning Env

In [ ]:
class NeuralNetwork(nn.Module):
    """
    Creates the individual MLPs
    """
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.out = nn.Linear(64, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.out(x)


class DeepRL:
    """
    Contains all of the elements related to deel QRL
    """
    def __init__(self, device):
        self.device = device
        self.batch_size = 256

        """
        Major improvement from model which had 8 outputs of the four actions 
        and four bets. Model wasted a lot of capacity learning the rules of the game and did not
        perform well. Creating seperate models, buffers, and epsilon scheduling made a huge difference
        """

        #Bet Model: hidden_prob, true_count_norm
        self.bcModel = NeuralNetwork(11, 4).to(device); self.bnModel = NeuralNetwork(11, 4).to(device); self.bnModel.eval()

        #Action Model: Takes hidden_prob, stake, player_total_norm, is_soft, dealer_up, true_count_norm
        self.acModel = NeuralNetwork(15, 4).to(device); self.anModel = NeuralNetwork(15, 4).to(device); self.anModel.eval()

        # Create optimizers and loss funciton
        self.bet_optimizer = torch.optim.Adam(self.bcModel.parameters(),lr=0.00001 )
        self.act_optimizer = torch.optim.Adam(self.acModel.parameters(), lr=0.0001)
        self.loss_fn = nn.MSELoss()

        #Create Buffers
        self.bet_buffer = deque(maxlen=200000); self.act_buffer = deque(maxlen=200000)

        # Action Epsilon Scheduling
        self.epsilon_action = 0.20; self.epsilon_decay_action = 0.999999455; self.min_epsilon_action = 0

        #Bet Scheduling
        self.epsilon_bet = 1.0; self.epsilon_decay_bet = 0.99999933; self.min_epsilon_bet = 0.05

        #Remaining Variables to Set
        self.discount = 0.99; self.step_count = 0;self.update_freq = 5000

## Import Training Display

In [ ]:
def display(epsilon_history_bet, epsilon_history_action, avg_return, loss_action, loss_bet, cfm):
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))

    x = np.linspace(0, len(epsilon_history_bet), len(epsilon_history_bet))
    axs[0, 0].plot(x, epsilon_history_action, color='red', label="Action Epsilon")
    axs[0, 0].plot(x, epsilon_history_bet, color='green', label="Bet Epsilon")
    axs[0, 0].set_title('Epsilon History')
    axs[0, 0].set_xlabel('Episodes')
    axs[0, 0].set_ylabel('Epsilon Value')
    axs[0,0].legend()
    axs[0, 0].grid(True)

    x = np.linspace(0, len(loss_action), len(loss_action))
    axs[1, 0].plot(x, loss_action, color='red', label="Loss Action Model")
    axs[1, 0].plot(x, loss_bet, color='green', label="Loss Bet Model")
    axs[1, 0].set_title('Loss History')
    axs[1, 0].set_xlabel('Episodes')
    axs[1, 0].set_ylabel('Loss')
    axs[1,0].legend()
    axs[1, 0].grid(True)

    x = np.linspace(0, len(avg_return),len(avg_return))
    axs[0, 1].plot(x, avg_return, color='green')
    axs[0, 1].set_title('Average Reward History (0.1-10 Bet Range)')
    axs[0, 1].set_xlabel('Episodes')
    axs[0, 1].set_ylabel('Avg Reward')
    axs[0, 1].grid(True)

    df = pd.DataFrame(cfm)

    sns.heatmap(df, cmap="YlGnBu", ax=axs[1,1])

    axs[1, 1].set_title("Game Result Map", fontsize=15)
    axs[1, 1].set_xlabel("Dealer Up", fontsize=12)
    axs[1, 1].set_ylabel("Player Up", fontsize=12)

    plt.tight_layout()
    plt.show()


### Display Containers

I chose to average over a period of rewards and loss so that it would appear clearly in the graphs. Without this element, the graphs were very hard to decipher.

In [ ]:
reward_history = [0]

loss_bet_container = deque(maxlen=1000)
loss_action_container = deque(maxlen=1000)
loss_history_action = []
loss_history_bet = []

reward_holder = deque(maxlen=10000)
res_player = deque(maxlen=1000)
res_dealer = deque(maxlen=1000)
epsilon_history_bet = []
epsilon_history_action = []

### Display Helper Functions

In [ ]:
def update_display(game, loss_action, loss_bet):
    reward_holder.appendleft(game.reward())

    tm = average(reward_holder)
    reward_history.append(tm)

    loss_action_container.appendleft(loss_action)
    loss_bet_container.appendleft(loss_bet)

    loss_history_action.append(average(loss_action_container))
    loss_history_bet.append(average(loss_bet_container))

    epsilon_history_bet.append(deepRL.epsilon_bet)
    epsilon_history_action.append(deepRL.epsilon_action)

    for each in game.hands: #added case for splits
        res_player.appendleft(game.get_score_and_soft(each)[0])
        res_dealer.appendleft(game.get_score_and_soft(game.dealer)[0])


def count_display():
    deepRL.step_count += 1
    if deepRL.step_count % deepRL.update_freq == 0:
        
        deepRL.anModel.load_state_dict(deepRL.acModel.state_dict())
        deepRL.bnModel.load_state_dict(deepRL.bcModel.state_dict())

        clear_output()
        cfm = np.zeros((max(res_player)+1, max(res_dealer)+1))
        for m, n in zip(res_player, res_dealer):
            cfm[m,n] = cfm[m,n] + 1
        display(epsilon_history_bet, epsilon_history_action, reward_history, loss_history_action, loss_history_bet, cfm)
        print(f"Step: {deepRL.step_count} | Avg Reward: {average(reward_holder):.2f} | Epsilon: {deepRL.epsilon_bet:.2f} {deepRL.epsilon_action:.2f}")

## Train Model

### Per Episode Handler

In [ ]:
def run_episode(game):

    """
    Create deck and run through a game while saving critical information to buffer
    """

    game.run_game(deepRL)

    reward_holder.appendleft(game.reward())

    #greedy epsilon decay
    deepRL.epsilon_action = max(deepRL.epsilon_action * deepRL.epsilon_decay_action, deepRL.min_epsilon_action)
    deepRL.epsilon_bet = max(deepRL.epsilon_bet * deepRL.epsilon_decay_bet, deepRL.min_epsilon_bet)

    if len(deepRL.bet_buffer) < deepRL.update_freq and len(deepRL.act_buffer) < deepRL.update_freq:
        return

    """
    Formatting buffer data w action buffer
    """
    batch = random.sample(deepRL.act_buffer, deepRL.batch_size)
    states, actions, next_states, rewards, dones = zip(*batch)
    states = torch.cat(states).to(device)
    next_states = torch.cat(next_states).to(device)
    actions = torch.tensor(actions, dtype=torch.long, device=device).unsqueeze(1)
    rewards = torch.tensor(rewards, dtype=torch.float, device=device)
    dones = torch.tensor(dones, dtype=torch.float, device=device)

    """
    Bellman Optimality Deep RL
    """
    y_hat = deepRL.acModel(states).gather(1, actions).squeeze()

    with torch.no_grad():
        next_actions = deepRL.acModel(next_states).argmax(dim=1, keepdim=True)
        max_next_q = deepRL.anModel(next_states).gather(1, next_actions).squeeze()

        y = rewards + (deepRL.discount * max_next_q * (1 - dones))

    loss_value_action = deepRL.loss_fn(y_hat, y)

    deepRL.act_optimizer.zero_grad()
    loss_value_action.backward()
    deepRL.act_optimizer.step()

    """
    Formatting buffer data w bet buffer
    """
    batch = random.sample(deepRL.bet_buffer, deepRL.batch_size)
    states, actions, rewards = zip(*batch)
    states = torch.cat(states).to(device)
    actions = torch.tensor(actions, dtype=torch.long, device=device).unsqueeze(1)
    rewards = torch.tensor(rewards, dtype=torch.float, device=device)

    """
    Bellman Optimality Deep RL
    """
    y_hat = deepRL.bcModel(states).gather(1, actions).squeeze()

    y = rewards

    loss_value_bet = deepRL.loss_fn(y_hat, y)

    deepRL.bet_optimizer.zero_grad()
    loss_value_bet.backward()
    deepRL.bet_optimizer.step()

    """
    Display related programming
    """
    update_display(game, loss_value_action.item(), loss_value_bet.item()) #update display content
    count_display() #on certain count update display/refresh what you see

    return

## Visualize Training

In [ ]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

deepRL = DeepRL(device)

####
#Below is for loading and continuing to train saved models
####

#deepRL.bcModel.load_state_dict(torch.load("bet.pth", weights_only=True, map_location=torch.device(device)))
#deepRL.bnModel.load_state_dict(torch.load("bet.pth", weights_only=True, map_location=torch.device(device)))
#deepRL.acModel.load_state_dict(torch.load("action.pth", weights_only=True, map_location=torch.device(device)))
#deepRL.anModel.load_state_dict(torch.load("action.pth", weights_only=True, map_location=torch.device(device)))

### Episode Loop

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    game = Deck(6, 0.87, device)
    for i in range(600000):
        run_episode(game)

## Save Models

In [ ]:
torch.save(deepRL.bcModel.state_dict(), "Models/random_bet.pth")
torch.save(deepRL.acModel.state_dict(), "Models/random_action.pth")

## Test Model Performance at Upper-Limit

In [ ]:
####
#Make Networks
####
bet_model = NeuralNetwork(11, 4)
action_model = NeuralNetwork(15, 4)

####
#Load Models
####
bet_model.load_state_dict(torch.load("Models/bet.pth", weights_only=True, map_location=torch.device(device)))
action_model.load_state_dict(torch.load("Models/action.pth", weights_only=True, map_location=torch.device(device)))

####
#Prepare for Evaluation
####
bet_model.to(device)
action_model.to(device)
bet_model.eval()
action_model.eval()

### Testing Helper

In [ ]:
profit_history = [0]
outcomes = [0,0,0]
result_history = []
res_dict = {}
count_history = []
total_bet = []

def run_episode(game):
    game.run_test(bet_model, action_model, count_history, total_bet)
    ret = game.reward()
    profit_history.append(profit_history[-1] + ret)
    result_history.append(ret)
    if ret not in res_dict.keys():
        res_dict[ret] = 0
    res_dict[ret] += 1

    if ret < 0:
        outcomes[-1] += 1
    elif ret > 0:
        outcomes[0] += 1
    else:
        outcomes[1] += 1

### Test Model

In [ ]:
game = Deck(6, 0.87, device)
start = time.time()
runs = 10000000 #Testing Model Toward Upper Bound

print("Progress Bar: ")
f = IntProgress(min=0, max=runs)
display_bar(f)

for i in range(runs):
    f.value = i
    time.sleep(0.0005)
    run_episode(game)

fig, axs = plt.subplots(2, 2, figsize=(12, 12))

x = np.linspace(0, len(profit_history), len(profit_history))
axs[0, 1].set_title("Profit History")
axs[0, 1].plot(x, profit_history, color = "blue")
axs[0,1].set_ylabel("Return")
axs[0,1].set_xlabel("Episodes")

axs[1,0].set_title("Normalized Outcomes Per Run")
axs[1, 0].bar(["Win", "Draw", "Lose"], [outcomes[0]/sum(outcomes), outcomes[1]/sum(outcomes), outcomes[2]/sum(outcomes)], color = "blue")
axs[1,0].set_ylabel("Percent")

items = sorted(res_dict.items(), key=lambda x: x[0])

labels = [str(round(k, 2)) for k, _ in items]
total = sum(v for _, v in items)
values = [v / total for _, v in items]

axs[0,0].set_title("Normalized Results Per Run")
axs[0, 0].bar(labels, values, color="blue")
axs[0, 0].set_xticklabels(labels, rotation=45, ha='right')
axs[0,0].set_ylabel("Percent")

pairs = sorted(count_history)                    
x_vals = [tc for tc, bet in pairs]
y_vals = [bet for tc, bet in pairs]

axs[1,1].scatter(x_vals, y_vals, alpha=0.1, s=4, color="blue")

axs[1,1].set_ylabel("Bet Value")
axs[1,1].set_xlabel("True Count")
axs[1,1].set_title("Bet vs True Count (Policy)")
axs[1,1].set_ylim(-0.5, 11)
axs[1,1].legend()
plt.show()

print(f"The average return per game was: {average(result_history):.6f} | The average runtime was: {(time.time()- start)/runs:.6f} seconds | The edge was :{(profit_history[-1]/sum(total_bet))* 100:2f} percent")